In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip

# TFT — Walmart Store Sales Forecasting

In [ ]:
!pip install "numpy<2" "torchvision==0.17.0" "torch==2.2.0" "neuralforecast==1.7.4" optuna mlflow dagshub wandb -q --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import wandb
import mlflow
import mlflow.pyfunc

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.auto import AutoTFT
from neuralforecast.losses.pytorch import MAE
from ray import tune

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'TFT'

MLFLOW_EXPERIMENT = 'TFT_Training'
MLFLOW_MODEL_NAME = 'tft_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4
INPUT_SIZE = 52
N_TRIALS   = 5
VAL_START  = '2012-04-01'

wandb.login()

print('Setup OK')

## 1. Pre-processing

In [ ]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'TFT_Preprocessing',
)

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows' : df.shape[0],
    'raw_cols' : df.shape[1],
    'stores'   : df['Store'].nunique(),
    'depts'    : df['Dept'].nunique(),
    'date_min' : str(df['Date'].min().date()),
    'date_max' : str(df['Date'].max().date()),
})

null_pct = df.isnull().mean().mul(100).round(2)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ['column', 'null_pct']
wandb.log({'null_percentages': wandb.Table(dataframe=null_df)})
print('Nulls (%):')
print(null_df.to_string(index=False))

# ── Fill nulls in covariate columns ───────────────────────────
# TFT uses covariates so we need to handle nulls in feature columns.
# MarkDown columns have ~65% nulls — fill with 0 (no markdown active)
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df[markdown_cols] = df[markdown_cols].fillna(0)
# CPI and Unemployment: fill with store-level forward fill then global median
df[['CPI', 'Unemployment']] = (
    df.groupby('Store')[['CPI', 'Unemployment']]
    .transform(lambda x: x.ffill().bfill())
)
df[['CPI', 'Unemployment']] = df[['CPI', 'Unemployment']].fillna(df[['CPI', 'Unemployment']].median())

wandb.log({
    'nulls_after_fill': int(df.isnull().sum().sum()),
})

# ── TFT-specific: add calendar covariates ─────────────────────
# TFT can consume known future covariates — calendar features are
# known in advance so they are the cleanest input for TFT.
df['week']        = df['Date'].dt.isocalendar().week.astype(int)
df['month']       = df['Date'].dt.month
df['is_holiday']  = df['IsHoliday'].astype(int)

# ── Format for neuralforecast ──────────────────────────────────
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)

# Covariates TFT will use:
# - hist_exog: observed in the past only (markdowns, temperature, fuel)
# - futr_exog: known for the future too (calendar features, IsHoliday)
# - stat_exog: static per series (store type, store size)
HIST_EXOG = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
             'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
FUTR_EXOG = ['week', 'month', 'is_holiday']
STAT_EXOG = ['Type', 'Size']

# Encode store Type (A/B/C) as integer
df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})

keep_cols = ['unique_id', 'Date', 'Weekly_Sales'] + HIST_EXOG + FUTR_EXOG + STAT_EXOG
df_nf = (
    df[keep_cols]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

wandb.log({
    'hist_exog_cols' : HIST_EXOG,
    'futr_exog_cols' : FUTR_EXOG,
    'stat_exog_cols' : STAT_EXOG,
})

min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'   : len(series_len),
    'valid_series'   : len(valid_ids),
    'dropped_series' : len(dropped),
    'min_series_len' : int(series_len[valid_ids].min()),
    'max_series_len' : int(series_len[valid_ids].max()),
})

print(f'\nValid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'train_date_min' : str(df_train['ds'].min().date()),
    'train_date_max' : str(df_train['ds'].max().date()),
    'val_date_min'   : str(df_val['ds'].min().date()),
    'val_date_max'   : str(df_val['ds'].max().date()),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'lookback_weeks' : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

## 2. Training

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def evaluate(nf: NeuralForecast, pred_col: str) -> tuple:
    preds   = nf.predict(df=df_val)
    eval_df = df_val.merge(
        preds.rename(columns={pred_col: 'y_pred'}),
        on=['unique_id', 'ds'], how='inner'
    )
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['is_holiday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [ ]:
# ── Baseline: single TFT with sensible defaults ────────────────
# TFT is the most powerful model here because it uses covariates:
# markdowns, temperature, fuel price, CPI, calendar features.
# The trade-off is longer training time vs N-BEATS and DLinear.
run_baseline = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TFT_Baseline',
    config   = {
        'input_size'    : INPUT_SIZE,
        'h'             : H,
        'hidden_size'   : 64,
        'n_head'        : 4,
        'dropout'       : 0.1,
        'max_steps'     : 500,
        'learning_rate' : 1e-3,
        'batch_size'    : 32,
        'loss'          : 'MAE',
        'hist_exog'     : HIST_EXOG,
        'futr_exog'     : FUTR_EXOG,
        'stat_exog'     : STAT_EXOG,
    }
)

baseline_model = TFT(
    h               = H,
    input_size      = INPUT_SIZE,
    hidden_size     = 64,
    n_head          = 4,
    dropout         = 0.1,
    hist_exog_list  = HIST_EXOG,
    futr_exog_list  = FUTR_EXOG,
    stat_exog_list  = STAT_EXOG,
    loss            = MAE(),
    max_steps       = 500,
    learning_rate   = 1e-3,
    batch_size      = 32,
    val_check_steps = 50,
    logger          = True,
)

nf_baseline = NeuralForecast(models=[baseline_model], freq='W-FRI')
nf_baseline.fit(df=df_train, val_size=len(df_val['ds'].unique()))

baseline_wmae, baseline_mae, eval_baseline = evaluate(nf_baseline, 'TFT')

wandb.log({
    'val_wmae' : baseline_wmae,
    'val_mae'  : baseline_mae,
})

print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Baseline MAE  : {baseline_mae:,.2f}')

wandb.finish()

In [ ]:
# ── AutoTFT: hyperparameter search ────────────────────────────
# TFT's most impactful parameters are hidden_size (model capacity)
# and n_head (attention heads — how many temporal patterns it can track).
run_search = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TFT_HPSearch',
    config   = {
        'n_trials'    : N_TRIALS,
        'h'           : H,
        'search_space': {
            'input_size'  : [26, 52, 104],
            'hidden_size' : [32, 64, 128, 256],
            'n_head'      : [2, 4, 8],
            'dropout'     : '0.0 – 0.3',
            'learning_rate': '1e-4 – 1e-3 (log)',
            'batch_size'  : [16, 32, 64],
        }
    }
)

tft_config = {
    'input_size'    : tune.choice([26, 52, 104]),
    'hidden_size'   : tune.choice([32, 64, 128, 256]),
    'n_head'        : tune.choice([2, 4, 8]),
    'dropout'       : tune.uniform(0.0, 0.3),
    'learning_rate' : tune.loguniform(1e-4, 1e-3),
    'batch_size'    : tune.choice([16, 32, 64]),
    'max_steps'     : 500,
}

auto_model = AutoTFT(
    h               = H,
    config          = tft_config,
    num_samples     = N_TRIALS,
    loss            = MAE(),
)

# TFT needs exog lists passed at fit time, not in config
nf_auto = NeuralForecast(models=[auto_model], freq='W-FRI')
nf_auto.fit(df=df_train, val_size=len(df_val['ds'].unique()))

best_wmae, best_mae, eval_best = evaluate(nf_auto, 'AutoTFT')
best_params = auto_model.results.get_best_result().config
trials_df   = auto_model.results.get_dataframe()

wandb.log({
    'best_val_wmae'    : best_wmae,
    'best_val_mae'     : best_mae,
    'baseline_wmae'    : baseline_wmae,
    'wmae_improvement' : baseline_wmae - best_wmae,
    'best_params'      : str(best_params),
    'trials_completed' : N_TRIALS,
    'all_trials'       : wandb.Table(dataframe=trials_df),
})

print(f'Best WMAE     : {best_wmae:,.2f}')
print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Improvement   : {baseline_wmae - best_wmae:,.2f}')
print(f'Best params   : {best_params}')

# Prediction plots
sample_ids = eval_best['unique_id'].unique()[:6]
fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
axes       = axes.flatten()

for ax, uid in zip(axes, sample_ids):
    history = df_train[df_train['unique_id'] == uid].tail(52)
    actual  = eval_best[eval_best['unique_id'] == uid]
    ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
    ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
    ax.plot(actual['ds'],  actual['y_pred'], label='TFT',     color='tomato', linestyle='--')
    hol = actual[actual['is_holiday'] == 1]
    ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
    ax.set_title(f'Store-Dept {uid}', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('TFT Best Model — Validation Predictions', fontsize=13)
plt.tight_layout()
wandb.log({'prediction_plots': wandb.Image(fig)})
plt.show()

per_series = (
    eval_best
    .groupby('unique_id')
    .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['is_holiday'].values))
    .reset_index()
    .rename(columns={0: 'wmae'})
    .sort_values('wmae', ascending=False)
)
wandb.log({'per_series_wmae': wandb.Table(dataframe=per_series)})

wandb.finish()

## 3. Save Best Model to MLflow Registry

In [ ]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

In [ ]:
class TFTWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['nf_model'], 'rb') as f:
            self.nf         = pickle.load(f)
            self.hist_exog  = pickle.load(f)
            self.futr_exog  = pickle.load(f)
            self.stat_exog  = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw merged DataFrame with:
        [Store, Dept, Date, Weekly_Sales, MarkDown1-5,
         Temperature, Fuel_Price, CPI, Unemployment,
         IsHoliday, Type, Size]
        Returns [Store, Dept, Date, Weekly_Sales_pred].
        """
        df_in = model_input.copy()
        df_in['Date']         = pd.to_datetime(df_in['Date'])
        df_in['unique_id']    = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)
        df_in['week']         = df_in['Date'].dt.isocalendar().week.astype(int)
        df_in['month']        = df_in['Date'].dt.month
        df_in['is_holiday']   = df_in['IsHoliday'].astype(int)
        markdown_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
        df_in[markdown_cols]  = df_in[markdown_cols].fillna(0)
        df_in['Type']         = df_in['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df_nf_in = (
            df_in
            .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
            .sort_values(['unique_id', 'ds'])
        )
        preds    = self.nf.predict(df=df_nf_in)
        pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
        preds    = preds.rename(columns={pred_col: 'Weekly_Sales_pred'})
        preds[['Store', 'Dept']] = preds['unique_id'].str.split('_', expand=True).astype(int)
        return preds[['Store', 'Dept', 'ds', 'Weekly_Sales_pred']].rename(columns={'ds': 'Date'})


os.makedirs('mlflow_artifacts', exist_ok=True)
model_path = 'mlflow_artifacts/nf_auto_tft.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(nf_auto, f)

with mlflow.start_run(run_name='TFT_Best_Model') as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'val_wmae'         : best_wmae,
        'val_mae'          : best_mae,
        'baseline_wmae'    : baseline_wmae,
        'wmae_improvement' : baseline_wmae - best_wmae,
        'n_trials'         : N_TRIALS,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'tft_model',
        python_model          = TFTWrapper(),
        artifacts             = {'nf_model': model_path},
        registered_model_name = MLFLOW_MODEL_NAME,
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best WMAE : {best_wmae:,.2f}')

In [ ]:
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
# TFT needs the full merged dataframe including covariates
sample = df[df['Store'] == 1].head(100).copy()
result = loaded.predict(sample)
print('Registry load OK')
print(result.head())